In [1]:
# Part 1: Import necessary libraries
import sqlite3
import sentencepiece as spm
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
print("Imports successful!")

c:\Users\Andrew\Desktop\Romeo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!


In [2]:
# Example: How to load the saved tensors and use them with the self-attention model
# This cell is for demonstration purposes and shouldn't be run until the previous cell has completed

"""
# Load the saved tensors
train_sequences = torch.load('cwe_train_sequences.pt')
train_labels = torch.load('cwe_train_labels.pt')
val_sequences = torch.load('cwe_val_sequences.pt')
val_labels = torch.load('cwe_val_labels.pt')
test_sequences = torch.load('cwe_test_sequences.pt')
test_labels = torch.load('cwe_test_labels.pt')

# Create datasets and dataloaders
train_dataset = TensorDataset(train_sequences, train_labels)
val_dataset = TensorDataset(val_sequences, val_labels)
test_dataset = TensorDataset(test_sequences, test_labels)

batch_size = 40
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

# Then you can use these loaders with your self-attention model similar to how it's done in self-attention.py
"""

"\n# Load the saved tensors\ntrain_sequences = torch.load('cwe_train_sequences.pt')\ntrain_labels = torch.load('cwe_train_labels.pt')\nval_sequences = torch.load('cwe_val_sequences.pt')\nval_labels = torch.load('cwe_val_labels.pt')\ntest_sequences = torch.load('cwe_test_sequences.pt')\ntest_labels = torch.load('cwe_test_labels.pt')\n\n# Create datasets and dataloaders\ntrain_dataset = TensorDataset(train_sequences, train_labels)\nval_dataset = TensorDataset(val_sequences, val_labels)\ntest_dataset = TensorDataset(test_sequences, test_labels)\n\nbatch_size = 40\ntrain_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)\nval_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)\ntest_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)\n\n# Then you can use these loaders with your self-attention model similar to how it's done in self-attention.py\n"

In [3]:
# Constants
MAX_FUNCTION_LINES = 50
MIN_FUNCTION_LINES = 10
NUM_CLASSES = 40
MAX_SEQ_LENGTH = 98
MAX_SENTENCES = 40

# Custom pad_sequences function to replace Keras dependency
def pad_sequences(sequences, maxlen, padding='post', value=0):
    """Pads sequences to the same length."""
    output = []
    for seq in sequences:
        if len(seq) > maxlen:
            # Truncate
            new_seq = seq[:maxlen]
        else:
            # Pad
            pad_length = maxlen - len(seq)
            if padding == 'post':
                new_seq = seq + [value] * pad_length
            else:  # 'pre'
                new_seq = [value] * pad_length + seq
        output.append(new_seq)
    return np.array(output)

print("Constants and padding function defined successfully!")

Constants and padding function defined successfully!


In [4]:
# Database functions
def load_data_from_db(db_path, limit_per_class=None):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Get functions with vulnerabilities
        cursor.execute(
            "SELECT cve, file, start, end, vuln, code FROM funcs WHERE vuln IS NOT NULL AND vuln != '' "
            f"AND (end - start + 1) BETWEEN {MIN_FUNCTION_LINES} AND {MAX_FUNCTION_LINES}"
        )
        vulnerable_funcs = cursor.fetchall()
        
        # Get functions without vulnerabilities
        cursor.execute(
            "SELECT cve, file, start, end, vuln, code FROM funcs WHERE (vuln IS NULL OR vuln = '') "
            f"AND (end - start + 1) BETWEEN {MIN_FUNCTION_LINES} AND {MAX_FUNCTION_LINES}"
        )
        non_vulnerable_funcs = cursor.fetchall()
        
        # Balance the dataset
        if limit_per_class and len(vulnerable_funcs) > limit_per_class:
            vulnerable_funcs = vulnerable_funcs[:limit_per_class]
        if limit_per_class and len(non_vulnerable_funcs) > limit_per_class:
            non_vulnerable_funcs = non_vulnerable_funcs[:limit_per_class]
        
        conn.close()
        
        # Combine the data
        all_funcs = vulnerable_funcs + non_vulnerable_funcs
        
        return all_funcs
    except Exception as e:
        print(f"Error in load_data_from_db: {str(e)}")
        return []

# Process vulnerability information to create labels
def create_labels(functions):
    data = []
    labels = []
    
    try:
        for cve, file, start, end, vuln, code in functions:
            code_lines = code.split('\n')
            
            # Skip if too many lines
            if len(code_lines) > MAX_FUNCTION_LINES:
                continue
            
            data.append(code)
            
            # Create one-hot encoded vector for vulnerability location
            # If vuln is None or empty, all zeros (no vulnerability)
            label = torch.zeros(NUM_CLASSES)
            
            if vuln:
                try:
                    # Parse the vulnerability line(s)
                    vuln_lines = [int(v.strip()) for v in vuln.split(',')]
                    for v_line in vuln_lines:
                        # Convert from absolute line number to relative position in function
                        rel_line = v_line - start
                        # Ensure it's within bounds
                        if 0 <= rel_line < NUM_CLASSES:
                            label[rel_line] = 1
                except Exception as e:
                    print(f"Error parsing vulnerability line: {str(e)}")
                    # In case of parsing errors, treat as non-vulnerable
                    pass
            
            labels.append(label)
    except Exception as e:
        print(f"Error in create_labels: {str(e)}")
    
    return data, labels

print("Database functions defined successfully!")

Database functions defined successfully!


In [5]:
# Load a small sample from the databases to test
try:
    print("Loading data from databases (small sample)...")
    # Use a small limit to test first
    c_funcs = load_data_from_db("c_10+.db", limit_per_class=10)
    java_funcs = load_data_from_db("java_10+.db", limit_per_class=10)
    
    print(f"Loaded {len(c_funcs)} C functions and {len(java_funcs)} Java functions")
    
    if len(c_funcs) > 0:
        print("\nSample C function metadata:")
        print(f"CVE: {c_funcs[0][0]}")
        print(f"File: {c_funcs[0][1]}")
        print(f"Start line: {c_funcs[0][2]}")
        print(f"End line: {c_funcs[0][3]}")
        print(f"Vulnerability: {c_funcs[0][4]}")
        print(f"Code snippet (first 100 chars): {c_funcs[0][5][:100]}...")
    
    # Combine data from both languages
    all_funcs = c_funcs + java_funcs
    data, labels = create_labels(all_funcs)
    
    print(f"\nProcessed {len(data)} code samples with labels")
    if len(labels) > 0:
        print(f"Sample label: {labels[0]}")
        # Count non-zero elements to check if vulnerabilities exist
        print(f"Number of vulnerabilities in sample: {torch.sum(labels[0]).item()}")
    
except Exception as e:
    print(f"Error during data loading: {str(e)}")

Loading data from databases (small sample)...
Loaded 20 C functions and 20 Java functions

Sample C function metadata:
CVE: CWE-114: Process Control
File: CWE114_Process_Control__w32_char_connect_socket_42.c
Start line: 117
End line: 138
Vulnerability: 10
Code snippet (first 100 chars): void CWE114_Process_Control__w32_char_connect_socket_42_bad()
{
    char * data;
    char dataBuf...

Processed 40 code samples with labels
Sample label: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Number of vulnerabilities in sample: 0.0


In [ ]:
### Load the tokenizer model from hugging face transformers library 
tokenizer = AutoTokenizer.from_pretrained("aiXcoder/aixcoder-7b-base")

c:\Users\Andrew\Desktop\Romeo\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Andrew\.cache\huggingface\hub\models--aiXcoder--aixcoder-7b-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular H

In [ ]:
# Test tokenization using Hugging Face tokenizer
try:
    print("Testing tokenization with Hugging Face tokenizer...")
    
    # Test on first sample
    sample_text = data[0]
    lines = sample_text.split('\n')[:3]  # Take first 3 lines for test
    
    print(f"Sample text (first line): {lines[0][:50]}..." if len(lines[0]) > 50 else lines[0])
    
    # Tokenize the first line
    tokens = tokenizer.encode(lines[0])
    print(f"First line tokenized to {len(tokens)} tokens")
    
    # Test padding function
    tokenized_lines = [tokenizer.encode(line) for line in lines]
    padded_lines = pad_sequences(tokenized_lines, maxlen=MAX_SEQ_LENGTH, padding='post')
    print(f"Padded shape: {padded_lines.shape}")
    
    # Pad to MAX_SENTENCES
    if len(padded_lines) < MAX_SENTENCES:
        padding = np.zeros((MAX_SENTENCES - len(padded_lines), MAX_SEQ_LENGTH))
        padded_lines = np.vstack((padded_lines, padding))
    
    print(f"Final padded shape: {padded_lines.shape}")
    
    # Convert to tensor
    tensor = torch.tensor(padded_lines)
    print(f"Tensor shape: {tensor.shape}")
    
except Exception as e:
    print(f"Error in tokenization test: {str(e)}")
    import traceback
    traceback.print_exc()

: 

In [ ]:
# Simplified tokenize_and_pad function using Hugging Face tokenizer
def tokenize_and_pad(data, max_samples=None):
    # Limit samples if needed
    if max_samples and len(data) > max_samples:
        data = data[:max_samples]
        print(f"Limited to {max_samples} samples")
    
    sequences = []
    
    for idx, text in enumerate(tqdm(data, desc="Tokenizing")):
        # Extract and limit lines
        lines = text.split('\n')[:MAX_SENTENCES]
        
        # Tokenize each line
        tokenized_lines = [tokenizer.encode(line) for line in lines]
        
        # Pad each line to MAX_SEQ_LENGTH
        padded_lines = pad_sequences(tokenized_lines, maxlen=MAX_SEQ_LENGTH, padding='post')
        
        # Pad to MAX_SENTENCES if needed
        if len(padded_lines) < MAX_SENTENCES:
            padding = np.zeros((MAX_SENTENCES - len(padded_lines), MAX_SEQ_LENGTH))
            padded_lines = np.vstack((padded_lines, padding))
        elif len(padded_lines) > MAX_SENTENCES:
            padded_lines = padded_lines[:MAX_SENTENCES]
        
        sequences.append(padded_lines)
    
    # Convert to tensor
    return torch.tensor(np.array(sequences))

# Test on a small subset
print("Testing tokenization with a small subset...")
if len(data) > 0:
    # Process just 2 samples
    test_sequences = tokenize_and_pad(data[:2], max_samples=2)
    print(f"Test sequences shape: {test_sequences.shape}")
    
    # Create a mini dataset
    test_labels_tensor = torch.stack(labels[:2])
    test_dataset = TensorDataset(test_sequences, test_labels_tensor)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    print("Dataset created successfully")

In [ ]:
# Streamlined dataset creation pipeline
# Step 1: Load data from databases
print("Loading data from databases...")
c_funcs = load_data_from_db("c_10+.db", limit_per_class=5000)  # Adjust limit as needed
java_funcs = load_data_from_db("java_10+.db", limit_per_class=5000)
print(f"Loaded {len(c_funcs)} C functions and {len(java_funcs)} Java functions")

# Combine data and create labels
all_funcs = c_funcs + java_funcs
data, labels = create_labels(all_funcs)
print(f"Processed {len(data)} code samples with labels")

# Step 2: Split the data
train_data, temp_data, train_labels, temp_labels = train_test_split(
    data, labels, test_size=0.3, random_state=42
)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.33, random_state=42
)
print(f"Train: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)} samples")

# Step 3: Tokenize and create tensors
print("Tokenizing data...")
train_sequences = tokenize_and_pad(train_data)
val_sequences = tokenize_and_pad(val_data)
test_sequences = tokenize_and_pad(test_data)

# Convert labels to tensors
train_labels_tensor = torch.stack(train_labels)
val_labels_tensor = torch.stack(val_labels)
test_labels_tensor = torch.stack(test_labels)

# Step 4: Create datasets and loaders
batch_size = 40
train_dataset = TensorDataset(train_sequences, train_labels_tensor)
val_dataset = TensorDataset(val_sequences, val_labels_tensor)
test_dataset = TensorDataset(test_sequences, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
print("DataLoaders created successfully!")

# Step 5: Visualize label distribution
label_counts = torch.sum(train_labels_tensor, dim=0).numpy()
plt.figure(figsize=(12, 6))
plt.bar(range(NUM_CLASSES), label_counts)
plt.xlabel('Line Position')
plt.ylabel('Number of Vulnerabilities')
plt.title('Distribution of Vulnerabilities by Line Position')
plt.tight_layout()
plt.show()

# Step 6: Save tensors
print("Saving tensors...")
torch.save(train_sequences, 'cwe_train_sequences.pt')
torch.save(train_labels_tensor, 'cwe_train_labels.pt')
torch.save(val_sequences, 'cwe_val_sequences.pt')
torch.save(val_labels_tensor, 'cwe_val_labels.pt')
torch.save(test_sequences, 'cwe_test_sequences.pt')
torch.save(test_labels_tensor, 'cwe_test_labels.pt')

print("Dataset creation complete!")

# Scaling Up to Full Dataset

After confirming that the code works with the smaller dataset, you can scale up by:

1. Increasing the `limit_per_class` parameter in the `load_data_from_db` calls
2. Removing the `max_samples` limit in tokenization
3. Increasing the batch size back to 40 if memory allows

This modular approach helps avoid kernel crashes by identifying issues at each step of the process.

# Memory-Efficient Dataset Creation

The cells above have been modified to be more memory-efficient and to provide better error diagnostics. Here are the key improvements:

1. **Better error handling** - Each step now has detailed error messages to pinpoint issues
2. **Memory optimization** - Processing one sample at a time and cleaning up after each step
3. **Fallback mechanisms** - When a step fails, the code continues with default values
4. **Shape validation** - Ensures output tensors are always the correct shape
5. **Progress reporting** - More detailed progress information during long operations

Try running the cells one by one, starting with cell 1 (imports) and proceeding through each step. This should help identify and work around the tokenization issues.

In [ ]:
# Single sample tokenization test with Hugging Face tokenizer
# Process one sample and save it
print("Testing with a single sample...")

# Load a minimal dataset
c_funcs = load_data_from_db("c_10+.db", limit_per_class=1)
print(f"Loaded {len(c_funcs)} C function")

if len(c_funcs) > 0:
    # Process just one sample
    sample_data, sample_label = create_labels(c_funcs)
    print(f"Processed 1 code sample with label")
    
    # Tokenize the single sample
    text = sample_data[0]
    lines = text.split('\n')[:MAX_SENTENCES]
    print(f"Sample has {len(lines)} lines")
    
    # Tokenize and pad
    tokenized_lines = [tokenizer.encode(line) for line in lines]
    padded_lines = pad_sequences(tokenized_lines, maxlen=MAX_SEQ_LENGTH, padding='post')
    
    # Pad to MAX_SENTENCES if needed
    if len(padded_lines) < MAX_SENTENCES:
        padding = np.zeros((MAX_SENTENCES - len(padded_lines), MAX_SEQ_LENGTH))
        padded_lines = np.vstack((padded_lines, padding))
    
    # Convert to tensor
    sequence_tensor = torch.tensor(padded_lines).unsqueeze(0)  # Add batch dimension
    print(f"Sequence tensor shape: {sequence_tensor.shape}")
    
    # Prepare label
    label_tensor = torch.stack(sample_label)
    print(f"Label tensor shape: {label_tensor.shape}")
    
    # Save single-sample tensors
    torch.save(sequence_tensor, 'cwe_single_sample_sequence.pt')
    torch.save(label_tensor, 'cwe_single_sample_label.pt')
    
    print("Single-sample test complete!")
else:
    print("No data available for testing")

In [ ]:
# Example of how to use the saved tensors with the self-attention model
"""
# Load the saved tensors
train_sequences = torch.load('cwe_train_sequences.pt')
train_labels = torch.load('cwe_train_labels.pt')
val_sequences = torch.load('cwe_val_sequences.pt')
val_labels = torch.load('cwe_val_labels.pt')
test_sequences = torch.load('cwe_test_sequences.pt')
test_labels = torch.load('cwe_test_labels.pt')

# Create datasets and DataLoaders
train_dataset = TensorDataset(train_sequences, train_labels)
val_dataset = TensorDataset(val_sequences, val_labels)
test_dataset = TensorDataset(test_sequences, test_labels)

batch_size = 40
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

# Load the self-attention model
from models.self-attention import LSTMClassifier

# Initialize model with appropriate parameters
model = LSTMClassifier(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=4096,  
    hidden_dim=256,
    output_dim=40,  
    n_layers=2,
    batch_first=True,
    bidirectional=True,
    dropout=0.5,
    pretrained_weights=None,  # You can use pre-trained embeddings if available
    batch_size=batch_size,
    sentence_length=MAX_SEQ_LENGTH
)

# Train the model
# optimizer = Adam(model.parameters(), lr=0.001)
# criterion = nn.BCELoss()
# train(model, train_loader, optimizer, criterion, device='cuda' if torch.cuda.is_available() else 'cpu')
"""